# Single-Embedding SCE на триплетах

Этот ноутбук реализует вариант под задачу:

- обучаемся на `train_triplets` (`anchor, positive, negative`);
- учим **общие паттерны совместимости** через SCE-подобные conditions;
- на инференсе получаем **один эмбеддинг на айтем**;
- совместимость считаем как `cosine(z_i, z_j)` без pairwise forward для каждой пары.


## 1) Импорты и базовые настройки


In [ ]:
from __future__ import annotations

from dataclasses import dataclass
from typing import Dict, Iterable

import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import CLIPModel


## 2) Конфиг лосса

`LossConfig` управляет балансом:

- `triplet_margin` — насколько `cos(a,p)` должен быть больше `cos(a,n)`;
- `pair_bce_weight` — вклад бинарной классификации пар (aux loss);
- `orthogonality_weight` — разнос condition-прототипов;
- `entropy_weight` — чтобы роутер не схлопывался в один condition.


In [ ]:
@dataclass
class LossConfig:
    triplet_margin: float = 0.2
    pair_bce_weight: float = 0.25
    orthogonality_weight: float = 1e-3
    entropy_weight: float = 1e-3


## 3) Модель: SingleEmbeddingSCENet

Идея похожа на SCE-Net, но с ключевым отличием:

- conditions используются при обучении,
- итогом всегда является один item-вектор `z`,
- значит, на инференсе можно кэшировать `item_id -> z`.

Формула:

\[
z = \mathrm{normalize}(z_{base} + s \cdot (\alpha \cdot C))
\]

где:
- `z_base` — базовый CLIP-вектор,
- `C` — матрица condition-прототипов,
- `alpha` — веса conditions, предсказанные item-only роутером.


In [ ]:
class SingleEmbeddingSCENet(nn.Module):
    def __init__(
        self,
        clip_model_name: str,
        num_conditions: int = 5,
        router_hidden_dim: int = 512,
        dropout: float = 0.1,
        cond_scale: float = 0.5,
    ) -> None:
        super().__init__()
        self.clip = CLIPModel.from_pretrained(clip_model_name)
        self.emb_dim = self._infer_emb_dim()
        self.num_conditions = num_conditions
        self.cond_scale = cond_scale

        # Condition prototypes: [M, D]
        self.condition_prototypes = nn.Parameter(torch.empty(num_conditions, self.emb_dim))
        nn.init.xavier_uniform_(self.condition_prototypes)

        # Item-only routing: z_base -> alpha in R^M
        self.router = nn.Sequential(
            nn.Linear(self.emb_dim, router_hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(router_hidden_dim, num_conditions),
        )

        # Auxiliary pair-BCE calibration over cosine
        self.pair_logit_scale = nn.Parameter(torch.tensor(10.0))
        self.pair_logit_bias = nn.Parameter(torch.tensor(0.0))

    def _infer_emb_dim(self) -> int:
        if hasattr(self.clip, "visual_projection") and self.clip.visual_projection is not None:
            proj = self.clip.visual_projection
            if hasattr(proj, "out_features"):
                return int(proj.out_features)
        if getattr(self.clip.config, "projection_dim", None) is not None:
            return int(self.clip.config.projection_dim)
        raise ValueError("Cannot infer image embedding dimension from CLIP model")

    def extract_base(self, pixel_values: torch.Tensor) -> torch.Tensor:
        z = self.clip.get_image_features(pixel_values=pixel_values)
        return F.normalize(z, dim=-1)

    def encode_with_conditions(self, z_base: torch.Tensor):
        alpha = F.softmax(self.router(z_base), dim=-1)  # [B, M]
        cond_vec = alpha @ self.condition_prototypes     # [B, D]
        z = F.normalize(z_base + self.cond_scale * cond_vec, dim=-1)
        return z, alpha

    def forward(self, pixel_values: torch.Tensor):
        z_base = self.extract_base(pixel_values)
        return self.encode_with_conditions(z_base)

    @staticmethod
    def cosine_score(z_left: torch.Tensor, z_right: torch.Tensor) -> torch.Tensor:
        return F.cosine_similarity(z_left, z_right, dim=-1)


## 4) Лосс для triplet training

Собираем итоговый objective:

1. `triplet`: хотим `cos(a,p)` > `cos(a,n)` + margin;
2. `pair BCE`: дополнительно калибруем positive/negative пары;
3. `orthogonality`: conditions должны быть различимыми;
4. `entropy`: роутер не должен всегда выбирать один condition.


In [ ]:
def compatibility_loss(
    model: SingleEmbeddingSCENet,
    z_anchor: torch.Tensor,
    z_pos: torch.Tensor,
    z_neg: torch.Tensor,
    alpha_anchor: torch.Tensor,
    alpha_pos: torch.Tensor,
    alpha_neg: torch.Tensor,
    cfg: LossConfig,
):
    cos_ap = F.cosine_similarity(z_anchor, z_pos, dim=-1)
    cos_an = F.cosine_similarity(z_anchor, z_neg, dim=-1)

    # cos(a,p) should be larger than cos(a,n)
    triplet = F.relu(cos_an - cos_ap + cfg.triplet_margin).mean()

    logits_pos = model.pair_logit_scale.clamp(1.0, 30.0) * cos_ap + model.pair_logit_bias
    logits_neg = model.pair_logit_scale.clamp(1.0, 30.0) * cos_an + model.pair_logit_bias
    pair_logits = torch.cat([logits_pos, logits_neg], dim=0)
    pair_targets = torch.cat([
        torch.ones_like(logits_pos),
        torch.zeros_like(logits_neg),
    ], dim=0)
    pair_bce = F.binary_cross_entropy_with_logits(pair_logits, pair_targets)

    proto = F.normalize(model.condition_prototypes, dim=-1)
    gram = proto @ proto.t()
    ident = torch.eye(gram.size(0), device=gram.device, dtype=gram.dtype)
    orth = ((gram - ident) ** 2).mean()

    alpha = torch.cat([alpha_anchor, alpha_pos, alpha_neg], dim=0)
    entropy = -(alpha * (alpha.clamp_min(1e-8).log())).sum(dim=-1).mean()

    loss = triplet + cfg.pair_bce_weight * pair_bce + cfg.orthogonality_weight * orth - cfg.entropy_weight * entropy

    stats = {
        "loss": float(loss.detach()),
        "triplet": float(triplet.detach()),
        "pair_bce": float(pair_bce.detach()),
        "orth": float(orth.detach()),
        "entropy": float(entropy.detach()),
        "cos_ap": float(cos_ap.mean().detach()),
        "cos_an": float(cos_an.mean().detach()),
    }
    return loss, stats


## 5) Шаг обучения на батче триплетов

Ожидается, что ваш `train_loader` отдаёт батч в формате:

```python
batch = {
  "anchor": {"pixel_values": ...},
  "positive": {"pixel_values": ...},
  "negative": {"pixel_values": ...},
}
```

Если у вас другой формат, просто замените доступ к тензорам в этой ячейке.


In [ ]:
def train_step(model, optimizer, batch, device, loss_cfg: LossConfig):
    model.train()
    optimizer.zero_grad(set_to_none=True)

    x_a = batch["anchor"]["pixel_values"].to(device, non_blocking=True)
    x_p = batch["positive"]["pixel_values"].to(device, non_blocking=True)
    x_n = batch["negative"]["pixel_values"].to(device, non_blocking=True)

    z_a, alpha_a = model(x_a)
    z_p, alpha_p = model(x_p)
    z_n, alpha_n = model(x_n)

    loss, stats = compatibility_loss(
        model=model,
        z_anchor=z_a,
        z_pos=z_p,
        z_neg=z_n,
        alpha_anchor=alpha_a,
        alpha_pos=alpha_p,
        alpha_neg=alpha_n,
        cfg=loss_cfg,
    )

    loss.backward()
    optimizer.step()
    return stats


## 6) Построение кэша эмбеддингов для инференса

После обучения делаем офлайн-проход по каталогу и сохраняем один эмбеддинг на айтем.


In [ ]:
@torch.no_grad()
def encode_items(model, dataloader: Iterable, device: torch.device, id_key: str = "item_id", pixel_key: str = "pixel_values") -> Dict[str, torch.Tensor]:
    model.eval()
    out: Dict[str, torch.Tensor] = {}

    for batch in dataloader:
        item_ids = batch[id_key]
        pixels = batch[pixel_key].to(device, non_blocking=True)
        z, _ = model(pixels)
        z = z.detach().cpu()
        for item_id, emb in zip(item_ids, z):
            out[str(item_id)] = emb
    return out


@torch.no_grad()
def cosine_compatibility(emb_index: Dict[str, torch.Tensor], left_item_id: str, right_item_id: str) -> float:
    a = emb_index[left_item_id]
    b = emb_index[right_item_id]
    return float(F.cosine_similarity(a.unsqueeze(0), b.unsqueeze(0), dim=-1).item())


## 7) Минимальный запуск (пример)

Этот блок показывает, как инициализировать модель и оптимизатор.

> В реальном проекте подставьте ваш `hf_model_name`, `train_loader`, `catalog_loader`.


In [ ]:
# Пример (не запускается без данных/окружения):
# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# model = SingleEmbeddingSCENet(clip_model_name="openai/clip-vit-base-patch32", num_conditions=5).to(device)
# optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)
# loss_cfg = LossConfig(triplet_margin=0.2)
#
# for batch in train_loader:
#     stats = train_step(model, optimizer, batch, device, loss_cfg)
#     print(stats)
#
# emb_index = encode_items(model, catalog_loader, device)
# score = cosine_compatibility(emb_index, "item_123", "item_987")
# print("compatibility cosine:", score)


## 8) Что важно мониторить

- `cos_ap` должен расти, `cos_an` — снижаться;
- triplet-loss и pair-BCE не должны конфликтовать (следите за весами);
- распределение `alpha` по conditions должно быть не деградировавшим;
- итоговые метрики: AUC/PR-AUC + Recall@K по кросс-категориям.

Если `alpha` всегда пиковый в один condition — увеличьте `entropy_weight`.
Если conditions слишком похожи — увеличьте `orthogonality_weight`.
